# 07 — Evaluation Pipeline, Calibration & AI Quality Scoring

**No API key required.** Comprehensive ML evaluation beyond accuracy.

Capabilities:
- Per-skill metrics: accuracy, F1, MAE, RMSE, R^2, latency percentiles
- Expected Calibration Error (ECE)
- Confidence calibration via isotonic regression and Platt scaling
- 5-dimension AI quality score (Accuracy, Calibration, Completeness, Consistency, Latency)
- Latency profiler for pipeline stages
- Data quality scoring and Parquet storage

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

## 7.1 — MetricsSuite: per-skill evaluation

In [ ]:
from chemvision.eval.metrics import MetricsSuite

suite = MetricsSuite()

# Simulate spectrum extraction results
rng = np.random.RandomState(42)
for _ in range(50):
    truth = f'{rng.uniform(20, 60):.1f}'
    noise = rng.normal(0, 0.5)
    pred_val = f'{float(truth) + noise:.1f}'
    correct = abs(float(pred_val) - float(truth)) < 0.3
    conf = rng.uniform(0.6, 1.0)
    suite.add('extract_spectrum', pred_val, truth, confidence=conf, latency_ms=rng.uniform(500, 3000))
    suite.add_numeric('extract_spectrum', float(pred_val), float(truth), confidence=conf)

# Simulate molecular structure results
for _ in range(30):
    correct = rng.random() > 0.2  # 80% accuracy
    suite.add('molecular_structure', 'CC=O' if correct else 'CCO', 'CC=O',
              confidence=rng.uniform(0.5, 0.95), latency_ms=rng.uniform(800, 4000))

result = suite.compute()

print(f'Overall accuracy: {result.overall_accuracy:.1%}')
print(f'Macro F1:         {result.macro_f1:.3f}')
print(f'Overall ECE:      {result.overall_ece:.3f}')
print()

import pandas as pd
rows = []
for name, m in result.per_skill.items():
    rows.append({
        'Skill': name,
        'N': m.n_samples,
        'Accuracy': f'{m.accuracy:.1%}',
        'ECE': f'{m.ece:.3f}' if m.ece else '—',
        'MAE': f'{m.mae:.2f}' if m.mae else '—',
        'RMSE': f'{m.rmse:.2f}' if m.rmse else '—',
        'R^2': f'{m.r_squared:.3f}' if m.r_squared else '—',
        'p50 (ms)': f'{m.latency_p50_ms:.0f}' if m.latency_p50_ms else '—',
        'p95 (ms)': f'{m.latency_p95_ms:.0f}' if m.latency_p95_ms else '—',
    })
pd.DataFrame(rows)

## 7.2 — Confidence calibration

In [ ]:
from chemvision.eval.calibration import ConfidenceCalibrator

# Simulate an overconfident model
rng = np.random.RandomState(7)
raw_confs = rng.uniform(0.6, 1.0, size=200).tolist()
correct = [1 if rng.random() < 0.5 else 0 for _ in raw_confs]  # 50% accuracy but high conf

# --- Isotonic regression ---
cal_iso = ConfidenceCalibrator(method='isotonic')
r_iso = cal_iso.fit(raw_confs, correct)

# --- Platt scaling ---
cal_platt = ConfidenceCalibrator(method='platt')
r_platt = cal_platt.fit(raw_confs, correct)

print('Isotonic regression:')
print(f'  ECE before: {r_iso.ece_before:.3f}')
print(f'  ECE after:  {r_iso.ece_after:.3f}')
print(f'  Improvement: {r_iso.improvement_pct:.1f}%')
print()
print('Platt scaling:')
print(f'  ECE before: {r_platt.ece_before:.3f}')
print(f'  ECE after:  {r_platt.ece_after:.3f}')
print(f'  Improvement: {r_platt.improvement_pct:.1f}%')

In [ ]:
# Reliability diagram: before vs after calibration
x_range = np.linspace(0.01, 0.99, 50)
cal_iso_vals = [cal_iso.calibrate(x) for x in x_range]
cal_platt_vals = [cal_platt.calibrate(x) for x in x_range]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration', alpha=0.5)
ax.plot(x_range, x_range, 'gray', alpha=0.3, label='Raw (uncalibrated)')
ax.plot(x_range, cal_iso_vals, 'b-', linewidth=2, label=f'Isotonic (ECE={r_iso.ece_after:.3f})')
ax.plot(x_range, cal_platt_vals, 'r-', linewidth=2, label=f'Platt (ECE={r_platt.ece_after:.3f})')
ax.set_xlabel('Raw confidence', fontsize=12)
ax.set_ylabel('Calibrated probability', fontsize=12)
ax.set_title('Confidence Calibration', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 7.3 — AI Quality Scorer (5 dimensions)

In [ ]:
from chemvision.eval.quality import AIQualityScorer

scorer = AIQualityScorer(target_latency_ms=3000)

rng = np.random.RandomState(42)
for _ in range(50):
    scorer.add_result(
        correct=rng.random() > 0.15,       # 85% accuracy
        confidence=rng.uniform(0.7, 0.95),
        completeness=rng.uniform(0.6, 1.0),
        latency_ms=rng.uniform(500, 4000),
    )

# Add consistency data
for i in range(10):
    scorer.add_consistency_pair(f'img_{i}', 'result_A')
    scorer.add_consistency_pair(f'img_{i}', 'result_A' if rng.random() > 0.1 else 'result_B')

report = scorer.score()
print(report.summary())

In [ ]:
# Radar chart of quality dimensions
dims = report.dimensions
labels = [d.name for d in dims]
values = [d.score / d.max_score for d in dims]
values.append(values[0])  # close the polygon

angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles.append(angles[0])

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles, values, alpha=0.25, color='steelblue')
ax.plot(angles, values, 'o-', color='steelblue', linewidth=2)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title(f'AI Quality: {report.composite_score:.0f}/100 (Grade {report.grade})', fontsize=13, pad=20)
ax.grid(True)
plt.tight_layout()
plt.show()

## 7.4 — Latency profiler

In [ ]:
import time
from chemvision.eval.profiler import LatencyProfiler
from chemvision.models.mol_encoder import MolecularEncoder
from chemvision.generation.property_predictor import PropertyPredictor
from chemvision.physics.symmetry import CrystalSymmetryAnalyzer

profiler = LatencyProfiler()
enc = MolecularEncoder()
pred_fn = PropertyPredictor(use_mace=False)
sym_fn = CrystalSymmetryAnalyzer()

smiles_list = ['CC(=O)Oc1ccccc1C(=O)O', 'CCO', 'c1ccccc1', 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'] * 5

for smi in smiles_list:
    with profiler.measure('encode'):
        enc.encode(smi)
    with profiler.measure('conformer'):
        enc.generate_conformer(smi)
    with profiler.measure('predict'):
        pred_fn.predict(smi)

for _ in range(20):
    with profiler.measure('symmetry'):
        sym_fn.from_lattice_params(3.615, 3.615, 3.615, 90, 90, 90,
                                   species=[29]*4,
                                   fractional_positions=[[0,0,0],[.5,.5,0],[.5,0,.5],[0,.5,.5]])

report_prof = profiler.report()

rows = [{'Stage': r.stage, 'Calls': r.n_calls,
         'Mean (ms)': f'{r.mean_ms:.1f}', 'p50': f'{r.p50_ms:.1f}',
         'p95': f'{r.p95_ms:.1f}', 'Total (ms)': f'{r.total_ms:.0f}'}
        for r in report_prof]
pd.DataFrame(rows)

In [ ]:
# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 4))
stages = [r.stage for r in report_prof]
totals = [r.total_ms for r in report_prof]
bars = ax.barh(stages, totals, color='steelblue', edgecolor='navy')
ax.set_xlabel('Total time (ms)')
ax.set_title('Pipeline latency breakdown')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
for bar, v in zip(bars, totals):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{v:.0f} ms', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 7.5 — Data pipeline: Parquet store with quality scoring

In [ ]:
from chemvision.data.pipeline import DataStore, DataRecord
from pathlib import Path

store = DataStore(Path('/tmp/demo_store'))

# Ingest some molecules
molecules = [
    ('CCO', 'molecular', 'pubchem', 46.07),
    ('CC(=O)O', 'molecular', 'pubchem', 60.05),
    ('c1ccccc1', 'molecular', 'synthetic', 78.11),
    ('INVALID', 'molecular', 'test', None),
]

for smi, domain, source, mw in molecules:
    record = DataRecord(
        smiles=smi, domain=domain, source=source,
        molecular_weight=mw,
        provenance={'notebook': '07_eval_and_quality', 'purpose': 'demo'},
    )
    stored = store.ingest(record)
    print(f'{smi:15s} -> quality={stored.quality_score:.3f}  flags={stored.quality_flags}')

print(f'\nTotal records: {store.count()}')
print(f'Stats: {store.stats()}')